# 06 - Streaming Behavioral Feature Engineering

This notebook demonstrates personalized cardholder behavioral baselines, sliding time windows (1-min, 10-min, 24-hr), and strictly causal feature extraction.

In [1]:
import os, sys
from datetime import datetime, timedelta
sys.path.insert(0, os.path.abspath('..'))
from src.features.behavioral_features import CardBehaviorProfile
from src.features.behavior_score import BehaviorScorer

# Create card profile
profile = CardBehaviorProfile('CARD_DEMO')
t0 = datetime.now() - timedelta(hours=3)

# Feed historical normal transactions
for i in range(5):
    profile.update({
        'transaction_id': f'TX_HIST_{i}',
        'card_id': 'CARD_DEMO',
        'amount': 50.0 + (i * 5.0),
        'merchant': 'Target',
        'location': 'New York',
        'timestamp': (t0 + timedelta(minutes=i * 25)).isoformat()
    })
print(f'Recorded {len(profile.history)} historical events for CARD_DEMO')

Recorded 5 historical events for CARD_DEMO


In [2]:
# Extract features for normal transaction
current_normal = {
    'transaction_id': 'TX_CURR_NORM',
    'card_id': 'CARD_DEMO',
    'amount': 65.0,
    'merchant': 'Target',
    'location': 'New York',
    'timestamp': datetime.now().isoformat()
}
feats_norm = profile.get_features_for_transaction(current_normal)
scorer = BehaviorScorer()
score_norm, reasons_norm = scorer.calculate(feats_norm)
print('Normal Transaction Behavior Score:', score_norm)
print('Normal Features:', feats_norm)

Normal Transaction Behavior Score: 0
Normal Features: {'avg_amount': 60.0, 'max_normal_amount': 77.68, 'amount_ratio': 1.08, 'usual_hours': [16, 17, 18], 'is_unusual_hour': False, 'frequent_merchants': ['Target'], 'is_unusual_merchant': False, 'frequent_locations': ['New York'], 'is_unusual_location': False, 'txns_last_5min': 1, 'distinct_locations_last_1h': 1, 'avg_time_gap_seconds': 1500.0, 'daily_spend_last_24h': 365.0, 'typical_daily_spending': 300.0}


In [3]:
# Extract features for abnormal transaction (Spike + Unseen Merchant + Unseen Location)
current_anom = {
    'transaction_id': 'TX_CURR_ANOM',
    'card_id': 'CARD_DEMO',
    'amount': 550.0,  # ~9x customer average
    'merchant': 'LuxuryJewelry',
    'location': 'Tokyo',
    'timestamp': datetime.now().isoformat()
}
feats_anom = profile.get_features_for_transaction(current_anom)
score_anom, reasons_anom = scorer.calculate(feats_anom)
print('Anomalous Transaction Behavior Score:', score_anom)
print('Triggered Behavioral Reasons:')
for r in reasons_anom:
    print(' ->', r)

Anomalous Transaction Behavior Score: 0.72
Triggered Behavioral Reasons:
 -> Amount deviation: Amount is 9.2x the customer's average spending.
 -> Unusual merchant: Cardholder has never or rarely visited this merchant.
 -> Unusual location: Transaction origin differs from customary transaction locations.
 -> Daily surge: 24h spending ($850.00) is 2.8x normal daily average.


## Notebook Explanation

### What was done
- Instantiated personalized cardholder behavioral baselines (`CardBehaviorProfile`).
- Computed personalized statistics (mean amount, normal max, active hours, frequent merchants, frequent locations).
- Implemented sliding window aggregations (1-minute bursts, 10-minute velocity, 24-hour spend).
- Verified strict temporal causality: features use only transactions recorded prior to the evaluated event.
- Evaluated `BehaviorScorer` to convert behavioral deviations into a normalized [0.0, 1.0] risk score with explainable reasons.

### Why it was done
Global fraud rules fail to detect fraud for individual customer habits. A $500 purchase is normal for a corporate executive but highly anomalous for a student. Personalized behavioral baselines provide customized anomaly detection.

### Input
- Historical transactions and incoming transaction events.

### Output
- Personalized feature vector and normalized behavioral anomaly score.
- Human-readable behavioral anomaly explanations.

### Connection to next stage
The behavioral anomaly score feeds directly into `07_hybrid_fraud_detection.ipynb` to combine with the supervised ML score and rule engine score.

### Implementation decisions
- Cold-start handling was incorporated so first-time cards establish a baseline safely.
- Strict temporal filtering prevents future transactions from leaking into historical averages.